# Первинний огляд наборів даних

Цей блокнот перевіряє підготовлені CSV та показує описові показники поточного зрізу Києва. Тривоги походять з історичного джерела, погода є модельними даними для однієї точки, а **доставки синтетичні**. Місцевий час первинних тривог `Europe/Kyiv` прийнято для проєкту; підготовлені часові мітки подано в UTC.

Потрібні залежності проєкту та локальні файли з `data/processed/` і `data/synthetic/`. Блокнот використовує ядро з `.venv`. Блокнот лише читає дані; він не перезаписує CSV і не підключається до бази. Деталі джерел та обмежень — у [контрактах даних](../docs/03_data_contracts.md) і [підсумку дослідження](../docs/34_research_synthesis.md).

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Jupyter може бути відкритий із кореня проєкту або папки notebooks.
root = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / "scripts" / "load_dwh.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Відкрийте блокнот із кореня проєкту або папки notebooks")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.load_dwh import read_sources

# Використовуємо ту саму перевірку контрактів, що й завантажувач DWH.
datasets = read_sources(root)
frames = {name: pd.DataFrame(dataset.rows) for name, dataset in datasets.items()}
print("Контракти CSV перевірено. Кількість рядків:")
display(pd.DataFrame({name: len(frame) for name, frame in frames.items()}, index=["rows"]).T)

## Повнота та ключі

Таблиця показує кількість порожніх полів і повторних ключів. Завантажувач уже перевірив формат, часові мітки, допустимі значення та повноту погодинного ряду; цей огляд їх доповнює, а не перевіряє точність зовнішніх джерел.

In [ ]:
keys = {"alerts": "event_id", "deliveries": "delivery_id", "weather": "time"}
quality = []
for name, frame in frames.items():
    quality.append({"dataset": name, "rows": len(frame),
                    "empty_fields": int(frame.replace("", pd.NA).isna().sum().sum()),
                    "duplicate_keys": int(frame.duplicated(keys[name]).sum())})
display(pd.DataFrame(quality).set_index("dataset"))

## Тривалість тривог і модельні доставки

Дуже коротка подія не видаляється автоматично: її межі можна звірити з первинним файлом. Збіг із первинним файлом не доводить фактичної правильності джерела. Статуси доставок згенеровані й не вимірюють реальних затримок.

In [ ]:
alerts = frames["alerts"].copy()
alerts["started_at"] = pd.to_datetime(alerts["started_at"], utc=True)
alerts["finished_at"] = pd.to_datetime(alerts["finished_at"], utc=True)
alerts["duration_seconds"] = (alerts["finished_at"] - alerts["started_at"]).dt.total_seconds()
display(alerts["duration_seconds"].describe().to_frame())
print("Тривоги коротші за 60 секунд:")
display(alerts.loc[alerts["duration_seconds"] < 60,
                   ["event_id", "started_at", "finished_at", "duration_seconds"]])
print("Синтетичні статуси доставок:")
display(frames["deliveries"]["status"].value_counts().to_frame("deliveries"))

## Контрольна часова експозиція

Для кожної модельної доставки обчислюється сума секунд перетину з об'єднаними інтервалами тривог. Результат описує експозицію **синтетичного** набору, а не причинний вплив на фактичну логістику. Окрему поелементну перевірку в базі та сценарій демонстрації наведено в [документі 35](../docs/35_pgadmin_demo.md).

In [ ]:
from scripts.interval_join_baseline import indexed, merge, timestamp

intervals = merge([(timestamp(r["started_at"]), timestamp(r["finished_at"]))
                   for r in datasets["alerts"].rows])
operations = [(timestamp(r["started_at"]), timestamp(r["finished_at"]))
              for r in datasets["deliveries"].rows]
seconds = indexed(operations, intervals)
assert len(seconds) == len(operations)
assert all(0 <= value <= end - start for (start, end), value in zip(operations, seconds))
display(pd.DataFrame([{"synthetic_deliveries": len(seconds),
                       "deliveries_with_overlap": sum(value > 0 for value in seconds),
                       "total_overlap_seconds": sum(seconds)}]))